# Air Quality Index (AQI) Analysis & Forecasting – Indian Cities

**Dataset:** CPCB via Kaggle (`city_day.csv`)  
**Cities focus:** Delhi, Mumbai, Kolkata, Bengaluru  
**Models:** ARIMA & Facebook Prophet  
**Goal:** Uncover pollution trends, forecast next 6–12 months, recommend policy interventions

## 0. Install Dependencies

In [ ]:
# Run this cell once to install required libraries
import subprocess, sys
packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'statsmodels', 'prophet', 'scikit-learn']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('All packages installed.')

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error
from prophet import Prophet

# Plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 5), 'font.size': 11})

CITIES = ['Delhi', 'Mumbai', 'Kolkata', 'Bengaluru']
POLLUTANTS = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3']
FORECAST_CITY = 'Delhi'   # city used for time-series modelling
FORECAST_MONTHS = 12

## 2. Load & Preview Data

In [ ]:
df = pd.read_csv('city_day.csv', parse_dates=['Date'])
print(f'Shape: {df.shape}')
print(f'Cities: {df["City"].nunique()}  |  Date range: {df["Date"].min().date()} → {df["Date"].max().date()}')
df.head()

## 3. Data Cleaning & Preprocessing

In [ ]:
# --- Missing value summary ---
missing = df.isnull().mean().mul(100).round(2).sort_values(ascending=False)
print('Missing % per column:')
print(missing[missing > 0].to_string())

In [ ]:
# Visualise missing data heatmap
fig, ax = plt.subplots(figsize=(14, 4))
miss_pct = df[POLLUTANTS + ['AQI']].isnull().mean().mul(100)
ax.bar(miss_pct.index, miss_pct.values, color=sns.color_palette('muted', len(miss_pct)))
ax.set_ylabel('Missing (%)')
ax.set_title('Missing Data Percentage per Pollutant')
plt.tight_layout()
plt.show()

In [ ]:
# Filter to 4 focus cities
df = df[df['City'].isin(CITIES)].copy()

# Forward-fill then backward-fill missing pollutant values within each city
df.sort_values(['City', 'Date'], inplace=True)
df[POLLUTANTS + ['AQI']] = (
    df.groupby('City')[POLLUTANTS + ['AQI']]
      .transform(lambda s: s.ffill().bfill())
)

# Drop rows still missing AQI after fill
df.dropna(subset=['AQI'], inplace=True)
df.reset_index(drop=True, inplace=True)

# Add time features
df['Year']   = df['Date'].dt.year
df['Month']  = df['Date'].dt.month
df['Season'] = pd.cut(
    df['Month'],
    bins=[0, 2, 5, 8, 11, 12],
    labels=['Winter', 'Spring', 'Summer', 'Monsoon', 'Winter'],
    ordered=False
)

print(f'Clean dataset shape: {df.shape}')
df.describe().round(2)

## 4. Exploratory Data Analysis (EDA)

### 4.1 AQI Distribution by City

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Box plot
sns.boxplot(data=df, x='City', y='AQI', palette='Set2', ax=axes[0])
axes[0].set_title('AQI Distribution by City')
axes[0].set_xlabel('')

# Violin plot
sns.violinplot(data=df, x='City', y='AQI', palette='Set2', inner='quartile', ax=axes[1])
axes[1].set_title('AQI Violin Plot by City')
axes[1].set_xlabel('')

plt.suptitle('City-wise AQI Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 4.2 AQI Bucket Distribution

In [ ]:
bucket_order = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']
bucket_colors = ['#2ecc71', '#a9d18e', '#f1c40f', '#e67e22', '#e74c3c', '#8e44ad']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, city in zip(axes.flat, CITIES):
    city_df = df[df['City'] == city]
    counts = city_df['AQI_Bucket'].value_counts().reindex(bucket_order, fill_value=0)
    ax.bar(counts.index, counts.values,
           color=[bucket_colors[bucket_order.index(b)] for b in counts.index])
    ax.set_title(city, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('AQI Category Distribution by City', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.3 Yearly AQI Trend

In [ ]:
yearly = df.groupby(['City', 'Year'])['AQI'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
for city in CITIES:
    cdf = yearly[yearly['City'] == city]
    ax.plot(cdf['Year'], cdf['AQI'], marker='o', label=city, linewidth=2)

ax.set_title('Yearly Average AQI Trend by City', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Average AQI')
ax.legend()
plt.tight_layout()
plt.show()

### 4.4 Monthly / Seasonal Patterns

In [ ]:
monthly = df.groupby(['City', 'Month'])['AQI'].mean().reset_index()
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(13, 5))
for city in CITIES:
    cdf = monthly[monthly['City'] == city]
    ax.plot(cdf['Month'], cdf['AQI'], marker='o', label=city, linewidth=2)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)
ax.set_title('Monthly Average AQI – Seasonal Pattern', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Average AQI')
ax.legend()
plt.tight_layout()
plt.show()

### 4.5 Pollutant Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
corr = df[POLLUTANTS + ['AQI']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Pollutant Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.6 Average Pollutant Levels by City

In [ ]:
poll_city = df.groupby('City')[POLLUTANTS].mean().round(2)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
palette = sns.color_palette('Set2', len(CITIES))
for ax, pollutant in zip(axes.flat, POLLUTANTS):
    ax.bar(poll_city.index, poll_city[pollutant], color=palette)
    ax.set_title(pollutant, fontweight='bold')
    ax.set_ylabel('µg/m³')
    ax.tick_params(axis='x', rotation=20)

plt.suptitle('Average Pollutant Levels by City', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4.7 High-Risk Zones – Peak Pollution Days

In [ ]:
# Days with AQI > 300 ('Very Poor' / 'Severe')
high_risk = df[df['AQI'] > 300].groupby('City').size().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(high_risk.index, high_risk.values,
               color=sns.color_palette('Reds_r', len(high_risk)))
ax.bar_label(bars, padding=4)
ax.set_xlabel('Number of Days with AQI > 300')
ax.set_title('High-Risk Days (AQI > 300) by City', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop 10 worst pollution days:')
df[['City', 'Date', 'AQI', 'AQI_Bucket']].nlargest(10, 'AQI').to_string(index=False)

### 4.8 PM2.5 Rolling Average – Delhi

In [ ]:
delhi = df[df['City'] == 'Delhi'].set_index('Date').sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(delhi.index, delhi['PM2.5'], alpha=0.3, color='steelblue', label='Daily PM2.5')
ax.plot(delhi['PM2.5'].rolling(30).mean(), color='red', linewidth=2, label='30-day Rolling Mean')
ax.set_title('Delhi – PM2.5 Daily Values & 30-day Rolling Mean', fontsize=13, fontweight='bold')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Time Series Forecasting – Delhi AQI

We use **Delhi** as the forecasting city (highest AQI, most policy-relevant).

In [ ]:
# Monthly average AQI for Delhi (daily data is too noisy for ARIMA)
ts = (
    delhi['AQI']
    .resample('ME')          # month-end frequency
    .mean()
    .rename_axis('Date')
    .dropna()
)

# Train / test split: last 12 months = test
TRAIN_END = ts.index[-13]
train = ts[ts.index <= TRAIN_END]
test  = ts[ts.index >  TRAIN_END]

print(f'Train: {train.index.min().date()} → {train.index.max().date()}  ({len(train)} months)')
print(f'Test : {test.index.min().date()}  → {test.index.max().date()}  ({len(test)} months)')

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(train, label='Train', color='steelblue')
ax.plot(test,  label='Test',  color='orange')
ax.set_title('Delhi Monthly AQI – Train / Test Split', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

### 5.1 Stationarity Check (ADF Test)

In [ ]:
def adf_report(series, label=''):
    result = adfuller(series.dropna())
    p = result[1]
    print(f'{label}  ADF stat={result[0]:.4f}  p-value={p:.4f}  → {"STATIONARY" if p < 0.05 else "NON-STATIONARY"}')
    return p

p = adf_report(train, 'Original')
if p >= 0.05:
    adf_report(train.diff().dropna(), 'First diff')

### 5.2 ACF / PACF Plots

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(train.dropna(),  lags=24, ax=ax1, title='ACF – Delhi Monthly AQI')
plot_pacf(train.dropna(), lags=24, ax=ax2, title='PACF – Delhi Monthly AQI')
plt.tight_layout()
plt.show()

### 5.3 ARIMA Model

In [ ]:
# SARIMA(1,1,1)(1,1,1,12) captures seasonality
arima_model = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

print(arima_model.summary().tables[0])

In [ ]:
# Forecast on test period
arima_pred = arima_model.forecast(steps=len(test))
arima_pred.index = test.index

arima_mae  = mean_absolute_error(test, arima_pred)
arima_rmse = np.sqrt(mean_squared_error(test, arima_pred))
arima_mape = np.mean(np.abs((test.values - arima_pred.values) / test.values)) * 100

print(f'ARIMA  →  MAE={arima_mae:.2f}  RMSE={arima_rmse:.2f}  MAPE={arima_mape:.2f}%')

In [ ]:
# ARIMA future forecast (next 12 months)
arima_future = arima_model.forecast(steps=len(test) + FORECAST_MONTHS)
future_idx = pd.date_range(ts.index[-1], periods=FORECAST_MONTHS + 1, freq='ME')[1:]
arima_future_forecast = arima_model.forecast(steps=FORECAST_MONTHS)
arima_future_forecast.index = future_idx

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train,       label='Train',              color='steelblue')
ax.plot(test,        label='Actual (Test)',       color='orange')
ax.plot(arima_pred,  label='ARIMA Prediction',   color='red',    linestyle='--')
ax.plot(arima_future_forecast, label='ARIMA Future Forecast', color='purple', linestyle='-.')
ax.axvline(ts.index[-13], color='gray', linestyle=':', label='Train/Test split')
ax.set_title('ARIMA – Delhi AQI Forecast', fontsize=13, fontweight='bold')
ax.set_ylabel('AQI')
ax.legend()
plt.tight_layout()
plt.show()

### 5.4 Facebook Prophet Model

In [ ]:
# Prophet requires columns ds and y
prophet_train = train.reset_index().rename(columns={'Date': 'ds', 'AQI': 'y'})
prophet_test  = test.reset_index().rename(columns={'Date': 'ds', 'AQI': 'y'})

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='multiplicative',
    changepoint_prior_scale=0.1
)
prophet_model.fit(prophet_train)
print('Prophet model trained.')

In [ ]:
# Predict on test + future
future_df = prophet_model.make_future_dataframe(periods=len(test) + FORECAST_MONTHS, freq='ME')
prophet_forecast = prophet_model.predict(future_df)

# Test period predictions
prophet_test_pred = prophet_forecast[prophet_forecast['ds'].isin(test.index)]['yhat'].values

prophet_mae  = mean_absolute_error(test.values, prophet_test_pred)
prophet_rmse = np.sqrt(mean_squared_error(test.values, prophet_test_pred))
prophet_mape = np.mean(np.abs((test.values - prophet_test_pred) / test.values)) * 100

print(f'Prophet →  MAE={prophet_mae:.2f}  RMSE={prophet_rmse:.2f}  MAPE={prophet_mape:.2f}%')

In [ ]:
# Prophet plot
future_only = prophet_forecast[prophet_forecast['ds'] > ts.index[-1]]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index, train.values, label='Train',           color='steelblue')
ax.plot(test.index,  test.values,  label='Actual (Test)',   color='orange')
ax.plot(test.index,  prophet_test_pred, label='Prophet Prediction', color='green', linestyle='--')
ax.plot(future_only['ds'], future_only['yhat'], label='Prophet Future', color='darkgreen', linestyle='-.')
ax.fill_between(future_only['ds'],
                future_only['yhat_lower'],
                future_only['yhat_upper'],
                alpha=0.15, color='green', label='95% CI')
ax.axvline(ts.index[-13], color='gray', linestyle=':', label='Train/Test split')
ax.set_title('Prophet – Delhi AQI Forecast', fontsize=13, fontweight='bold')
ax.set_ylabel('AQI')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Prophet components (trend + seasonality)
fig = prophet_model.plot_components(prophet_forecast)
fig.suptitle('Prophet – Trend & Seasonality Components', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Model Comparison

In [ ]:
metrics = pd.DataFrame({
    'Model': ['ARIMA (SARIMA)', 'Facebook Prophet'],
    'MAE':   [arima_mae,  prophet_mae],
    'RMSE':  [arima_rmse, prophet_rmse],
    'MAPE%': [arima_mape, prophet_mape]
}).set_index('Model').round(2)

print('\n=== Model Comparison (lower is better) ===')
print(metrics.to_string())

# Determine winner
winner = metrics['RMSE'].idxmin()
print(f'\nBest model by RMSE: {winner}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric in zip(axes, ['MAE', 'RMSE', 'MAPE%']):
    colors = ['#e74c3c' if m == metrics[metric].idxmin() else '#95a5a6' for m in metrics.index]
    ax.bar(metrics.index, metrics[metric], color=colors)
    ax.set_title(metric, fontweight='bold')
    ax.tick_params(axis='x', rotation=10)
    for bar, val in zip(ax.patches, metrics[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('ARIMA vs Prophet – Performance Metrics\n(red = better)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted overlay – both models
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(test.index, test.values,        label='Actual',          color='black',  linewidth=2)
ax.plot(test.index, arima_pred.values,  label='ARIMA',           color='red',    linestyle='--')
ax.plot(test.index, prophet_test_pred,  label='Prophet',         color='green',  linestyle='-.')
ax.set_title('Actual vs Predicted – Test Period (Delhi AQI)', fontsize=13, fontweight='bold')
ax.set_ylabel('AQI')
ax.legend()
plt.tight_layout()
plt.show()

### Model Selection Recommendation

| Criterion | ARIMA (SARIMA) | Facebook Prophet |
|---|---|---|
| Handles seasonality | Yes (seasonal order) | Yes (built-in) |
| Handles holidays/events | No | Yes |
| Interpretability | Moderate | High (components plot) |
| Robustness to missing data | Low | High |
| Uncertainty intervals | Needs manual config | Built-in |
| Best for | Short stationary series | Long seasonal series with events |

**Decision rule (auto-printed below):**

In [ ]:
if prophet_rmse < arima_rmse:
    print(f'RECOMMENDATION: Use Facebook Prophet')
    print(f'  Prophet RMSE ({prophet_rmse:.2f}) < ARIMA RMSE ({arima_rmse:.2f})')
    print('  Prophet captures seasonality + provides confidence intervals — ideal for this use case.')
else:
    print(f'RECOMMENDATION: Use ARIMA (SARIMA)')
    print(f'  ARIMA RMSE ({arima_rmse:.2f}) < Prophet RMSE ({prophet_rmse:.2f})')
    print('  ARIMA performs better on this stationary monthly series.')

## 7. 12-Month Future Forecast (Best Model)

In [ ]:
# Retrain best model on full data, then forecast 12 months
if prophet_rmse <= arima_rmse:
    # Prophet on full data
    full_df = ts.reset_index().rename(columns={'Date': 'ds', 'AQI': 'y'})
    final_model = Prophet(yearly_seasonality=True, seasonality_mode='multiplicative',
                          weekly_seasonality=False, daily_seasonality=False,
                          changepoint_prior_scale=0.1)
    final_model.fit(full_df)
    future = final_model.make_future_dataframe(periods=FORECAST_MONTHS, freq='ME')
    final_forecast = final_model.predict(future)
    f_dates = final_forecast.tail(FORECAST_MONTHS)['ds']
    f_yhat  = final_forecast.tail(FORECAST_MONTHS)['yhat']
    f_lower = final_forecast.tail(FORECAST_MONTHS)['yhat_lower']
    f_upper = final_forecast.tail(FORECAST_MONTHS)['yhat_upper']
    model_label = 'Prophet'
else:
    # ARIMA on full data
    final_arima = SARIMAX(ts, order=(1,1,1), seasonal_order=(1,1,1,12),
                          enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    arima_fc = final_arima.get_forecast(steps=FORECAST_MONTHS)
    f_dates = pd.date_range(ts.index[-1], periods=FORECAST_MONTHS+1, freq='ME')[1:]
    f_yhat  = arima_fc.predicted_mean
    f_lower = arima_fc.conf_int()['lower AQI']
    f_upper = arima_fc.conf_int()['upper AQI']
    f_lower.index = f_dates
    f_upper.index = f_dates
    f_yhat.index  = f_dates
    model_label = 'ARIMA'

forecast_df = pd.DataFrame({'Date': f_dates, 'Forecast_AQI': f_yhat.values,
                             'Lower_CI': f_lower.values, 'Upper_CI': f_upper.values})
print(f'\n12-Month AQI Forecast for Delhi ({model_label}):')
print(forecast_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ts.index, ts.values, label='Historical AQI', color='steelblue', linewidth=1.5)
ax.plot(forecast_df['Date'], forecast_df['Forecast_AQI'],
        label=f'{model_label} Forecast', color='red', linewidth=2, linestyle='--')
ax.fill_between(forecast_df['Date'], forecast_df['Lower_CI'], forecast_df['Upper_CI'],
                alpha=0.2, color='red', label='95% Confidence Interval')
ax.axvline(ts.index[-1], color='gray', linestyle=':', alpha=0.8)
ax.set_title(f'Delhi AQI – 12-Month Forecast ({model_label})', fontsize=13, fontweight='bold')
ax.set_ylabel('AQI')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Key Insights & Policy Recommendations

In [ ]:
print('=' * 65)
print('         KEY INSIGHTS – INDIA AQI ANALYSIS')
print('=' * 65)

# Insight 1: Worst city
worst_city = df.groupby('City')['AQI'].mean().idxmax()
worst_aqi  = df.groupby('City')['AQI'].mean().max()
print(f'\n1. Most Polluted City: {worst_city} (avg AQI = {worst_aqi:.1f})')

# Insight 2: Worst month across all cities
worst_month = df.groupby('Month')['AQI'].mean().idxmax()
import calendar
print(f'2. Peak Pollution Month: {calendar.month_name[worst_month]}')

# Insight 3: Worst pollutant correlated with AQI
corr_with_aqi = df[POLLUTANTS].corrwith(df['AQI']).abs().idxmax()
print(f'3. Pollutant most correlated with AQI: {corr_with_aqi}')

# Insight 4: High-risk days count
severe_days = (df['AQI'] > 400).sum()
print(f'4. Total "Severe" AQI days (>400) across all cities: {severe_days}')

# Insight 5: YoY trend
yoy = df.groupby(['City','Year'])['AQI'].mean().unstack()
delhi_trend = yoy.loc['Delhi'].dropna()
trend_dir   = 'improving' if delhi_trend.iloc[-1] < delhi_trend.iloc[0] else 'worsening'
print(f'5. Delhi AQI trend: {trend_dir} over the dataset period')

In [ ]:
print('=' * 65)
print('      POLICY RECOMMENDATIONS')
print('=' * 65)
recommendations = [
    ('Vehicle Emission Control',
     'Enforce BS-VI norms city-wide. Expand odd-even vehicle rotation\n'
     '  during Oct–Jan peak season in Delhi and Kolkata.'),
    ('Green Zone Promotion',
     'Increase urban green cover in high-density zones. \n'
     '  Mandate green buffers around industrial clusters.'),
    ('Festival-based Emission Warnings',
     'Issue pre-Diwali and pre-harvest burn advisories. \n'
     '  Deploy emergency response teams during Nov–Dec.'),
    ('Industrial Regulation',
     'Migrate brick kilns and power plants to cleaner technology. \n'
     '  Real-time stack monitoring with public dashboards.'),
    ('Construction Dust Management',
     'Mandatory anti-smog guns and water sprinklers on all \n'
     '  construction sites above 500 sq.m in metro cities.'),
    ('Public Health Alerts',
     'Automated SMS/app alerts when AQI > 200. \n'
     '  Free mask distribution at metro stations during peak months.'),
]
for i, (title, detail) in enumerate(recommendations, 1):
    print(f'\n{i}. {title}:')
    print(f'   {detail}')

print('\n' + '=' * 65)

## 9. Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle('India AQI Analysis – Summary Dashboard', fontsize=16, fontweight='bold', y=1.01)

# Panel 1: City AQI distribution
ax1 = fig.add_subplot(2, 3, 1)
city_avg = df.groupby('City')['AQI'].mean().sort_values(ascending=False)
colors   = ['#e74c3c' if c == city_avg.idxmax() else '#3498db' for c in city_avg.index]
ax1.bar(city_avg.index, city_avg.values, color=colors)
ax1.set_title('Average AQI by City', fontweight='bold')
ax1.set_ylabel('AQI')

# Panel 2: Monthly seasonality
ax2 = fig.add_subplot(2, 3, 2)
for city in CITIES:
    m = df[df['City'] == city].groupby('Month')['AQI'].mean()
    ax2.plot(m.index, m.values, marker='o', label=city)
ax2.set_xticks(range(1,13))
ax2.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax2.set_title('Monthly AQI Pattern', fontweight='bold')
ax2.legend(fontsize=8)

# Panel 3: Model comparison RMSE
ax3 = fig.add_subplot(2, 3, 3)
mdl_names = ['ARIMA', 'Prophet']
rmse_vals = [arima_rmse, prophet_rmse]
bar_cols   = ['#e74c3c' if r == min(rmse_vals) else '#95a5a6' for r in rmse_vals]
ax3.bar(mdl_names, rmse_vals, color=bar_cols)
ax3.set_title('Model RMSE Comparison', fontweight='bold')
ax3.set_ylabel('RMSE')
for bar, val in zip(ax3.patches, rmse_vals):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}', ha='center', fontsize=10)

# Panel 4: Delhi AQI time series + forecast
ax4 = fig.add_subplot(2, 3, (4, 5))
ax4.plot(ts.index, ts.values, label='Historical', color='steelblue')
ax4.plot(forecast_df['Date'], forecast_df['Forecast_AQI'],
         label='Forecast', color='red', linestyle='--')
ax4.fill_between(forecast_df['Date'], forecast_df['Lower_CI'], forecast_df['Upper_CI'],
                 alpha=0.15, color='red')
ax4.set_title('Delhi AQI – Historical + 12-Month Forecast', fontweight='bold')
ax4.set_ylabel('AQI')
ax4.legend(fontsize=9)

# Panel 5: High-risk days
ax5 = fig.add_subplot(2, 3, 6)
hr = df[df['AQI'] > 300].groupby('City').size()
ax5.barh(hr.index, hr.values, color='#e74c3c')
ax5.set_title('High-Risk Days (AQI > 300)', fontweight='bold')
ax5.set_xlabel('Days')

plt.tight_layout()
plt.savefig('aqi_summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved as aqi_summary_dashboard.png')

---
## Project Complete

| Section | Status |
|---|---|
| Data Cleaning & Preprocessing | Done |
| Exploratory Data Analysis | Done |
| ARIMA Forecasting | Done |
| Prophet Forecasting | Done |
| Model Comparison | Done |
| 12-Month Forecast | Done |
| Policy Recommendations | Done |
| Summary Dashboard | Done |